# 用 Hugging Face Trainer 微调 Qwen2.5-0.5B-Instruct

这个 Notebook 将指令问答整理为因果语言建模样本，通过 PEFT 接入 LoRA，再由 Trainer 管理训练、验证和检查点。重点是理解手写训练循环中的数据、梯度更新和评估如何对应到框架接口。

**实验规模：6 条训练样本、2 条验证样本。** 这是微调流程练习；后面的 Loss 和少量生成结果不能用来判断模型的泛化能力。文中数值来自原有保存输出。

```text
本地模型与 Tokenizer → 对话模板 → Dataset.map 编码与标签屏蔽
  → Collator 动态补齐 → LoRA → Trainer 训练、验证、选择检查点
  → 生成检查 → 导出 Adapter → 新建基础模型并重载 Adapter → 再生成
```

## 1. 环境准备与库的分工

| 库 | 在本实验中的职责 |
|---|---|
| Transformers | 加载 Tokenizer 与 Qwen，提供生成接口、数据整理器、TrainingArguments 和 Trainer |
| Datasets | 将问答列表转成可索引的数据集，并用 `map()` 对两组样本执行相同的预处理 |
| PEFT | 给基础模型接入 LoRA，管理可训练参数及 Adapter 的保存和重载 |
| PyTorch | 提供张量、设备管理、前向和自动求导 |

代码先通过 `certifi` 配置 CA 证书，并在未显式传入证书来源时为默认 SSL 上下文提供证书文件。该替换在当前 Python 进程内生效，仍保留证书校验；“SSL 初始化成功”也不等于验证了远端连接。本节 SSL 修复属于当前实验环境的临时兼容处理，并非通用配置。其他环境如果不存在证书问题，不需要复制这段 SSL 修改代码。后续模型与 Tokenizer 明确只读本地文件或缓存。

版本检查输出记录了 Transformers 4.57.6、Datasets 4.5.0、PEFT 0.17.1 和 CUDA 可用。`inspect.signature()` 用于确认本环境支持 `Trainer(processing_class=...)` 和 `TrainingArguments(eval_strategy=...)`，避免把当前写法当成所有版本通用的接口。

In [ ]:
import os
import ssl
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

_original_create_default_context = ssl.create_default_context

def create_default_context_with_certifi(*args, **kwargs):
    if not any(kwargs.get(k) for k in ["cafile", "capath", "cadata"]):
        kwargs["cafile"] = certifi.where()

    return _original_create_default_context(*args, **kwargs)

ssl.create_default_context = create_default_context_with_certifi
context = ssl.create_default_context()
print("SSL 初始化成功，证书数量：", len(context.get_ca_certs()))

import sys
import inspect
import torch
import transformers
import datasets
import peft
from transformers import Trainer, TrainingArguments

SSLError: [ASN1: NOT_ENOUGH_DATA] not enough data (_ssl.c:4192)

In [2]:
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Current device: CPU (CUDA unavailable)")

trainer_params = inspect.signature(Trainer.__init__).parameters
args_params = inspect.signature(TrainingArguments.__init__).parameters

print("\nTrainer supports processing_class:",
      "processing_class" in trainer_params)

print("TrainingArguments supports eval_strategy:",
      "eval_strategy" in args_params)

Python: 3.9.25
PyTorch: 2.6.0+cu124
Transformers: 4.57.6
Datasets: 4.5.0
PEFT: 0.17.1

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Trainer supports processing_class: True
TrainingArguments supports eval_strategy: True


## 2. Tokenizer、基础模型与聊天模板

### 2.1 加载分词规则与特殊 Token

`AutoTokenizer.from_pretrained()` 根据模型标识加载配套的词表、分词规则、特殊 Token 和聊天模板。Token 是模型处理的文本片段，Token ID 是词表索引；其数值大小不代表语义远近。`local_files_only=True` 要求文件已在本地，缺失时不会自动下载。

| 保存输出 | 含义 |
|---|---|
| `len(tokenizer)=151665` | 当前 Tokenizer 的可用条目数，包含已添加 Token；不应直接当成模型输出层维度 |
| PAD：`<\|endoftext\|>`，ID 151643 | 用于批内长度对齐；有效位置由 Attention Mask 标记 |
| EOS：`<\|im_end\|>`，ID 151645 | 此配置的结束标记，在生成到指定 EOS 时可停止 |
| `padding_side="right"` | 补齐发生时，填充位置加在有效序列右侧；此属性本身不会触发补齐 |

PAD 和 EOS 在这里是两个不同 ID。模板中的角色边界也是模型输入的一部分，不能用普通文本拼接随意替代。

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True
)

print("词表大小:", len(tokenizer))
print("PAD Token:", tokenizer.pad_token)
print("EOS Token:", tokenizer.eos_token)
print("PAD Token ID:", tokenizer.pad_token_id)
print("EOS Token ID:", tokenizer.eos_token_id)
print("Padding Side:", tokenizer.padding_side)
print("是否存在 Chat Template:", tokenizer.chat_template is not None)

词表大小: 151665
PAD Token: <|endoftext|>
EOS Token: <|im_end|>
PAD Token ID: 151643
EOS Token ID: 151645
Padding Side: right
是否存在 Chat Template: True


### 2.2 加载基础模型与选择设备

`AutoModelForCausalLM` 加载已经预训练并做过指令对齐的 Qwen2.5-0.5B-Instruct，包含用于预测下一 Token 的语言模型头。此时还没有接入 LoRA，也没有更新权重。

代码在 CUDA 可用时选择 GPU 和 `torch.float16`，否则使用 CPU 和 `torch.float32`；`model.to(device)` 将模型移动到所选设备。历史输出记录为 `cuda:0`、FP16，参数量约 494.03M。这里的 dtype 控制加载时的权重类型，与后文 `TrainingArguments.fp16` 控制训练混合精度是两件事。初次加载的保存输出包含 `torch_dtype` 弃用提示，后面的重载代码实际使用了 `dtype`。

In [4]:
from transformers import AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.float16 if device.type == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype,
    local_files_only=True
)

model = model.to(device)

print("模型设备:", next(model.parameters()).device)
print("模型数据类型:", next(model.parameters()).dtype)
print("模型参数量:", sum(p.numel() for p in model.parameters()) / 1e6, "Million")

`torch_dtype` is deprecated! Use `dtype` instead!


模型设备: cuda:0
模型数据类型: torch.float16
模型参数量: 494.032768 Million


### 2.3 同一问题的训练文本和推理 Prompt

`messages` 是有顺序的消息列表，每条记录用 `role` 表示角色，用 `content` 保存文本。这里先用 `tokenize=False` 展示模板生成的**字符串**，真正的 Tokenization 在后面的预处理和生成阶段执行。保存输出中的默认 system 消息来自当前 Qwen 模板。

- **训练格式**：user 问题和 assistant 参考答案都已提供，`add_generation_prompt=False`，不额外追加下一轮回答开头。
- **推理格式**：只提供 user 问题，`add_generation_prompt=True` 追加 assistant 开头，交给模型继续生成，不把参考答案输入模型。

这里演示的参考答案是“15”，实际数据集第一条是“15。”；示例用于观察格式，训练目标以随后构造的数据集为准。

In [5]:
messages = [
    {"role": "user", "content": "7 + 8 等于多少？"},
    {"role": "assistant", "content": "15"}
]
train_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("训练文本：")
print(train_text)

inference_messages = [
    {"role": "user", "content": "7 + 8 等于多少？"}
]
inference_text = tokenizer.apply_chat_template(
    inference_messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n推理输入：")
print(inference_text)

训练文本：
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
7 + 8 等于多少？<|im_end|>
<|im_start|>assistant
15<|im_end|>


推理输入：
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
7 + 8 等于多少？<|im_end|>
<|im_start|>assistant



## 3. 构造训练集与验证集

`Dataset.from_list()` 将 Python 字典列表转换为可按行访问的数据集，原始列为 `question` 和 `answer`。两组数据是代码分别写出的，不是从外部下载或随机切分的：

- **训练集 6 条**：两道加法、两道首都问答和两道学习概念解释，用于参数更新。
- **验证集 2 条**：一道加法和一道首都问答，用于评估与选择检查点，不用于反向传播。

验证题与训练题主题相近，而且同一验证集会被每轮反复评估。它适合检查流程是否运作，既不等同于独立测试集，也无法支撑泛化结论。

In [6]:
from datasets import Dataset

train_data = [
    {"question": "7 + 8 等于多少？", "answer": "15。"},
    {"question": "12 + 5 等于多少？", "answer": "17。"},
    {"question": "日本的首都是哪里？", "answer": "日本的首都是东京。"},
    {"question": "法国的首都是哪里？", "answer": "法国的首都是巴黎。"},
    {"question": "请用一句话解释什么是机器学习。",
     "answer": "机器学习是让计算机从数据中学习规律，并利用这些规律进行预测或决策的方法。"},
    {"question": "什么是监督学习？",
     "answer": "监督学习是使用带有正确标签的数据训练模型，使模型学习输入与输出之间关系的方法。"}
]
eval_data = [
    {"question": "9 + 6 等于多少？", "answer": "15。"},
    {"question": "意大利的首都是哪里？", "answer": "意大利的首都是罗马。"}
]

train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

print("训练集：", train_dataset)
print("验证集：", eval_dataset)

print("\n训练集第一个样本：")
print(train_dataset[0])

训练集： Dataset({
    features: ['question', 'answer'],
    num_rows: 6
})
验证集： Dataset({
    features: ['question', 'answer'],
    num_rows: 2
})

训练集第一个样本：
{'question': '7 + 8 等于多少？', 'answer': '15。'}


## 4. Tokenization 与 Assistant-only Loss

### 4.1 从样本得到三列模型输入

`preprocess_function()` 为每条样本构造两份文本：只到 assistant 开头的 Prompt，以及包含参考答案的完整对话。由于字符串已经由模板加入所需标记，后续调用 Tokenizer 时使用 `add_special_tokens=False`，避免再额外添加特殊 Token。

完整对话设 `truncation=True, max_length=256`，**此时不 Padding**；Prompt 单独编码且没有应用相同的显式截断。代码检查 Prompt ID 是否恰好是完整 ID 的前缀，再按其长度划分监督边界。这检查的是 Token 序列，而非字符数。扩展到长样本时，要注意超长 Prompt 可能触发前缀不匹配错误，答案也可能被截断；现有代码未另行过滤没有有效答案目标的样本。

| 字段 | 用途 |
|---|---|
| `input_ids` | 完整对话的词表索引；问题与参考答案都参与前向 |
| `attention_mask` | 标记输入的有效位置；本阶段没有填充，之后 Collator 会给填充位置补 0 |
| `labels` | 与输入对齐的监督目标；Prompt 部分设为 `-100`，其余沿用完整输入中的 ID |

训练集和验证集分别通过 `Dataset.map()` 调用同一函数，移除原始文本列，只保留模型需要的三个字段。第一条样本三列长度都为 44；解码非 `-100` 的目标，可看到“15。<\|im_end\|>”和模板尾部换行。因而这里的 Assistant-only Loss 覆盖回答内容及其后保留的模板标记，不是只针对答案里的字词。

### 4.2 为什么屏蔽标签而不删除 Prompt

`-100` 是交叉熵的忽略标签，不是一个词表 ID。Prompt 仍留在输入里供模型理解问题，只是不把它的各个 Token 当成需要学习预测的目标。它与 `attention_mask=0` 不同：后者表示输入的填充位置，不承担标签筛选的职责。

因果 LM 内部做下一 Token 对齐：位置 $t$ 的 Logits $z_{b,t}$ 预测位置 $t+1$ 的标签 $y_{b,t+1}$。对有效目标集合 $\mathcal{S}=\{(b,t)\mid 0\le t<T-1,\ y_{b,t+1}\ne -100\}$，Token 级平均交叉熵可写为：

$$
\mathcal{L}=-\frac{1}{|\mathcal{S}|}\sum_{(b,t)\in\mathcal{S}}
\log\operatorname{softmax}(z_{b,t})_{y_{b,t+1}}.
$$

因此代码不需要手工把 labels 再移一位。最后一个 Prompt Token 的输出仍能预测第一个答案 Token；是否计入 Loss 取决于**被预测位置**的标签。

In [7]:
MAX_LENGTH = 256

def preprocess_function(example):
    prompt_messages = [
        {"role": "user", "content": example["question"]}
    ]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": example["answer"]}
    ]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    if input_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt 与完整训练文本不匹配！")

    labels = [-100] * len(prompt_ids) + input_ids[len(prompt_ids):]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [8]:
train_tokenized = train_dataset.map(
    preprocess_function,
    remove_columns=train_dataset.column_names
)
eval_tokenized = eval_dataset.map(
    preprocess_function,
    remove_columns=eval_dataset.column_names
)

print("训练集：", train_tokenized)
print("验证集：", eval_tokenized)

sample = train_tokenized[0]

print("\nInput IDs 长度：", len(sample["input_ids"]))
print("Attention Mask 长度：", len(sample["attention_mask"]))
print("Labels 长度：", len(sample["labels"]))

target_ids = [
    token_id for token_id in sample["labels"]
    if token_id != -100
]

print("\n参与 Loss 计算的文本：")
print(tokenizer.decode(target_ids))

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

训练集： Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6
})
验证集： Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})

Input IDs 长度： 44
Attention Mask 长度： 44
Labels 长度： 44

参与 Loss 计算的文本：
15。<|im_end|>



## 5. Data Collator 与动态 Padding

Dataset 保存的是单条变长样本；DataLoader 取出一批样本，再由 Collator 把它们整理为同形状的张量。这里使用 `DataCollatorForSeq2Seq` 的对齐功能，虽名字带 Seq2Seq，但没有传入编码器—解码器模型，也不构造 decoder 输入；实际训练仍是 decoder-only Causal LM。

| 阶段 | 本代码实际做法 |
|---|---|
| 样本预处理 | 对完整文本截断到最多 256 Token，保持变长，不补到 256；Prompt 的 labels 设为 `-100` |
| Batch 组装 | `padding=True` 按本批最长样本右侧补齐；输入补 PAD ID、mask 补 0、labels 补 `-100`，返回 PyTorch 张量 |

前缀屏蔽和尾部填充都可能产生 `-100`，但一个来自**真实上下文不参与监督**，另一个来自**占位位置不应成为目标**。在演示的双样本 Batch 中，第一条长度 44，补到 45 后末尾多了一个 `-100`，三个张量 shape 都是 `(2, 45)`。正式训练和验证的单设备 Batch Size 均为 1，这个演示不代表实际每步送入两条样本，也不代表每步都有填充。

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

features = [
    train_tokenized[0],
    train_tokenized[1]
]

batch = data_collator(features)

for key, value in batch.items():
    print(key, value.shape)

print("\nLabels:")
print(batch["labels"])

input_ids torch.Size([2, 45])
attention_mask torch.Size([2, 45])
labels torch.Size([2, 45])

Labels:
tensor([[  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,     16,     20,   1773, 151645,    198,   -100],
        [  -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,   -100,
           -100,   -100,   -100,   -100,     16,     22,   1773, 151645,    198]])


## 6. LoRA：冻结原权重，学习低秩增量

对输入向量 $x$ 和冻结的线性层权重 $W_0$，标准 LoRA 在忽略偏置和 Dropout 时可写为：

$$
y=W_0x+\frac{\alpha}{r}BAx,\qquad
A\in\mathbb{R}^{r\times d_{\mathrm{in}}},\quad B\in\mathbb{R}^{d_{\mathrm{out}}\times r}.
$$

$A$ 将输入投影到 $r$ 维，$B$ 再映射回输出维度。增量 $BA$ 的秩不超过 $r$；每层只训练 $r(d_{\mathrm{in}}+d_{\mathrm{out}})$ 个增量参数，无需更新完整的 $d_{\mathrm{in}}d_{\mathrm{out}}$ 权重矩阵。基础网络仍参与前向，不能把可训练参数比例直接当成显存或耗时比例。

| 配置 | 本实验的含义 |
|---|---|
| `r=8`、`lora_alpha=16` | 低秩瓶颈为 8，标准缩放 $\alpha/r=2$；这个缩放不是优化器学习率 |
| `target_modules` | 只匹配 Attention 的 `q_proj`、`k_proj`、`v_proj`、`o_proj`，不包含 MLP 投影 |
| `lora_dropout=0.05` | 训练时作用在 LoRA 分支输入上，评估模式关闭 |
| `bias="none"` | 不额外训练偏置，不意味着基础模型没有偏置 |
| `task_type="CAUSAL_LM"` | 按因果语言模型任务包装基础模型 |

`get_peft_model()` 注入 Adapter 并冻结本配置不训练的基础参数。保存输出为 **1,081,344 个可训练参数 / 495,114,112 个总参数（0.2184%）**，总数包含新增 Adapter。设置 `model.config.use_cache=False` 关闭此处训练前向的 KV 缓存配置；整段并行训练不依赖逐 Token 解码缓存。这既不是开启梯度检查点，也不是量化训练。

In [10]:
from peft import LoraConfig, get_peft_model

model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 7. TrainingArguments：将训练意图写成配置

### 7.1 批大小、梯度累积与更新步

`per_device_train_batch_size=1` 表示单设备一次前向处理一条样本；`gradient_accumulation_steps=4` 表示通常累积四个微批次的梯度后再执行一次优化器更新。单卡完整累积组的有效样本数是 $1\times4=4$，但这不意味着四条样本同时送进显存。验证批大小也是 1；验证不累积梯度。

本次单卡每轮有 6 个微批次。当前 Trainer 在 Epoch 末尾处理不满四批的余数，所以更新分组是 **4 + 2**，不是丢掉最后两条，也不是把余数带到下一轮。由此得到：

$$
\text{更新步数}=\left\lceil\frac{6}{4}\right\rceil\times10=20.
$$

这里的 Epoch 是一次数据集遍历，`global_step` 是 Trainer 的更新步计数，不是处理过的样本数或前向次数。

### 7.2 学习率、精度与运行设置

| 配置 | 为什么使用 |
|---|---|
| `learning_rate=2e-4` | 设置调度器使用的学习率基准；实际学习率随步数变化 |
| `warmup_ratio=0.1` | 按计划更新步计算预热，此处为 $\lceil20\times0.1\rceil=2$ 步；使用该版本默认线性调度，预热后逐步降低学习率 |
| `weight_decay=0.01` | 对适用的可训练参数组使用权重衰减；与交叉熵的标签屏蔽无关 |
| `fp16=(device.type == "cuda")` | 在 GPU 上启用 FP16 混合精度，训练后端管理自动混合精度与梯度缩放；不是把所有张量都转成 FP16 |
| `dataloader_num_workers=0` | 在主进程取数和组批，便于 Notebook 中的小规模练习 |
| `report_to="none"` | 不向外部实验追踪集成上报；Notebook 日志仍会显示 |

基础权重加载的 dtype 与训练混合精度分别控制**参数存储**和**部分运算的精度**。PEFT 默认还可将部分低精度 Adapter 参数转换为 FP32 以利于稳定训练；本 Notebook 没有逐一打印 Adapter 的 dtype。

### 7.3 日志、验证和检查点

- `logging_strategy="steps", logging_steps=1`：按每个更新步记录训练日志。
- `eval_strategy="epoch", save_strategy="epoch"`：每轮末尾验证并保存检查点；`output_dir` 是检查点目录，`save_total_limit=2` 限制通常保留的数量。
- `load_best_model_at_end=True`：结束时装回按指定指标选出的最佳检查点。此处使用 `eval_loss`，`greater_is_better=False`，即较低的验证 Loss 更优。这不等于早停，代码仍设定训练 10 轮。

In [11]:
training_args = TrainingArguments(
    output_dir="./qwen2.5-0.5b-lora-output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_strategy="steps",
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=(device.type == "cuda"),
    report_to="none",
    dataloader_num_workers=0
)

print("TrainingArguments 配置完成！")

TrainingArguments 配置完成！


## 8. Trainer 的组成与训练流程

`Trainer` 接收已经接入 LoRA 的模型、训练配置、训练/验证数据集与 Collator。`processing_class=tokenizer` 把配套 Tokenizer 交给 Trainer 管理，便于特殊 Token 对齐与保存；它不会替代此前的 `Dataset.map()`，也不自动决定 Assistant-only 标签。

`trainer.train()` 才启动训练。框架负责构建 DataLoader、创建优化器与学习率调度器、把 Batch 放到设备上、执行前向和反向、按组更新并在轮末评估和保存。可把手写循环对应为：

```text
取样与组批 → model(**batch) 得到 Loss → 反向与梯度累积
  → 更新优化器和学习率、清除梯度 → 写日志
  → Epoch 末无梯度评估 → 保存检查点 → 结束后恢复最佳模型
```

这里使用的是 Transformers 的通用 `Trainer`，不是 TRL 的 SFTTrainer；数据怎么分词、哪些标签参与 Loss，由前面的代码定义。基础参数被冻结，因此优化过程只更新可训练的 LoRA 参数。

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer 创建成功！")

Trainer 创建成功！


## 9. 阅读训练日志：区分三种 Loss

下方保留了原始训练表和完成日志。比较指标时，先确认各自的计算窗口：

| 指标 | 在本例中如何理解 |
|---|---|
| 表格的 Training Loss | Notebook 进度回调在评估时显示最近一条训练 Loss 日志；此处每更新步记录一次，表格值不是整轮平均 |
| 表格的 Validation Loss | 该轮训练结束后，在两条验证样本上以参考答案为目标计算的评估 Loss；评估不更新参数 |
| `train_result.training_loss` | Trainer 汇总整个训练过程的训练 Loss，保存输出为 **0.9205**；不是最后一步或最后一轮的 Loss |

保存输出记录 **20 个更新步**。训练日志会波动，例如第 9 轮显示 0.022900，第 10 轮显示 0.935700；累积组所含样本、参数状态和训练模式都可能影响这个值，现有日志不能单独确定波动的原因。验证 Loss 在这次保存运行中从第 1 轮的 **0.744583** 降到第 10 轮的 **0.022665**，最后一轮在这十次评估里最低。按配置，结束时选择的是验证 Loss 最低的检查点。

Loss 衡量模型给参考目标 Token 的概率；评估时先前的参考答案 Token 也可作为后续位置的上下文（teacher forcing），不同于让模型自由生成整段答案。这里验证集只有两条样本且题型接近训练集；不能把验证 Loss 下降换算成准确率，也不能据此判断泛化能力。

保存日志还记录了 Trainer 将模型与生成配置中的部分特殊 Token ID 对齐到 Tokenizer；这属于配置对齐，不是改变数据集里已经生成的标签。

In [13]:
train_result = trainer.train()

print("\n训练完成！")
print("总训练步数:", train_result.global_step)
print("平均训练 Loss:", f"{train_result.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,2.955500,0.744583
2,0.123700,0.714580
3,2.359300,0.623944
4,0.554900,0.493684
5,1.206000,0.256507
6,0.165300,0.124169
7,1.287400,0.104300
8,0.056700,0.067124
9,0.022900,0.034754
10,0.935700,0.022665



训练完成！
总训练步数: 20
平均训练 Loss: 0.9205


## 10. 训练后推理与文本生成

代码将模型设为 `eval()`，恢复 `model.config.use_cache=True`，并用 `torch.no_grad()` 关闭梯度记录。评估模式控制 Dropout 等行为，关闭梯度控制计算图；推理过程不做参数更新。KV 缓存用于生成时复用历史 Token 的 Key/Value，减少重复计算。

每题先把只有 user 的消息转成含 assistant 开头的 Prompt，再用 `return_tensors="pt"` 编码为张量并移动到模型设备。`**inputs` 传入 `input_ids`、`attention_mask` 等实际编码字段；没有传入 labels，也没有把标准答案塞进 Prompt。

| 生成参数 | 在此处的作用 |
|---|---|
| `max_new_tokens=64` | 最多续写 64 个 Token，不含输入 Prompt；遇到结束条件可提前停止 |
| `do_sample=False` | 关闭随机采样；常规单束配置下逐步选最高分候选，采样参数不会用于该选择 |
| `pad_token_id`、`eos_token_id` | 显式提供填充和结束标记，与当前 Tokenizer 对齐 |

返回序列包含原 Prompt，代码用输入宽度切掉前缀，只解码新增 ID，并用 `skip_special_tokens=True` 隐藏边界标记。原输出中的 `temperature`、`top_p`、`top_k` 提示与非采样设置相符，不表示生成失败。

三道题的保存输出依次为“15。”“15。”“日本的首都是东京。”；第一、三题来自训练集，第二题来自验证集。这是一组生成链路检查，没有独立测试数据，也没有同输入的基础模型对照。

In [14]:
model.eval()
model.config.use_cache = True

test_questions = [
    "7 + 8 等于多少？",
    "9 + 6 等于多少？",
    "日本的首都是哪里？"
]

for question in test_questions:
    messages = [
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    print("问题：", question)
    print("回答：", answer)
    print("-" * 50)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


问题： 7 + 8 等于多少？
回答： 15。
--------------------------------------------------
问题： 9 + 6 等于多少？
回答： 15。
--------------------------------------------------
问题： 日本的首都是哪里？
回答： 日本的首都是东京。
--------------------------------------------------


## 11. 导出 LoRA Adapter 与 Tokenizer

`model.save_pretrained(SAVE_DIR)` 保存当前 PEFT 模型的 Adapter，`tokenizer.save_pretrained(SAVE_DIR)` 另行保存分词与聊天模板资源。`./qwen2.5-0.5b-lora-final` 相对于 Notebook 内核工作目录，和 Trainer 的训练检查点目录用途不同。

| 资源 | 作用 |
|---|---|
| `adapter_config.json` | 记录基础模型标识、目标模块、秩与缩放等接入信息 |
| `adapter_model.safetensors` | 保存学习到的低秩参数；不是完整的 Qwen 权重 |
| `tokenizer.json`、`chat_template.jinja` 等 | 保存可在新进程重新使用的分词规则、特殊 Token 和对话格式 |

这里没有调用 `merge_and_unload()`，也没有导出可独立于基础模型运行的完整模型。该导出目录不等同于包含优化器、调度器和训练进度的 Trainer 检查点。

In [15]:
import os

SAVE_DIR = "./qwen2.5-0.5b-lora-final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("模型保存位置:", os.path.abspath(SAVE_DIR))
print("保存的文件:", os.listdir(SAVE_DIR))

模型保存位置: d:\LLM-learning\llm-finetuning\qwen2.5-0.5b-lora-final
保存的文件: ['adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## 12. 重新加载与独立模型实例的推理验证

先删除原有 Trainer 和训练模型引用，调用 `gc.collect()` 和 `torch.cuda.empty_cache()`；后者只释放可回收的缓存，不能释放仍由活跃张量占用的显存。随后以原模型标识和 dtype 从本地文件或缓存新建 `base_model_reload`，再用 `PeftModel.from_pretrained(base_model_reload, SAVE_DIR, is_trainable=False)` 将磁盘上的 Adapter 接回基础层。保存输出记录了已加载的 Adapter 名称 `default`。

这一步验证的是：**推理可用重新构建的基础模型和已保存 Adapter，不依赖 Trainer 或训练时的模型实例**。重载后用 `eval()` 和 `torch.inference_mode()` 执行生成；后者关闭自动求导并减少推理期间的额外跟踪。沿用同三道题、同一 Tokenizer 和同一显式生成参数（最多 64 个新 Token、非采样），便于比较重载前后打印的回答。

注意这里仍在**同一个 Notebook 内核**，复用了内存中的 `tokenizer`，没有启动独立进程，也没有从 `SAVE_DIR` 重载 Tokenizer。若在新进程推理，需要自行加载兼容的基础模型、导出目录的 Tokenizer 与 Adapter；这属于可延伸的验证，本 Notebook 未执行。

In [16]:
import gc
del trainer
del model

gc.collect()
torch.cuda.empty_cache()

base_model_reload = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=model_dtype,
    local_files_only=True
).to(device)

base_model_reload.eval()

print("基础模型重新加载成功！")
print("模型设备:", next(base_model_reload.parameters()).device)

基础模型重新加载成功！
模型设备: cuda:0


In [17]:
from peft import PeftModel

model_reload = PeftModel.from_pretrained(
    base_model_reload,
    SAVE_DIR,
    is_trainable=False
)

model_reload.eval()

print("LoRA Adapter 重新加载成功！")
print("模型设备:", next(model_reload.parameters()).device)
print("已加载的 Adapter:", list(model_reload.peft_config.keys()))

LoRA Adapter 重新加载成功！
模型设备: cuda:0
已加载的 Adapter: ['default']


In [18]:
test_questions = [
    "7 + 8 等于多少？",
    "9 + 6 等于多少？",
    "日本的首都是哪里？"
]

for question in test_questions:
    messages = [
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(device)

    with torch.inference_mode():
        outputs = model_reload.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    print("问题：", question)
    print("回答：", answer)
    print("-" * 50)

问题： 7 + 8 等于多少？
回答： 15。
--------------------------------------------------
问题： 9 + 6 等于多少？
回答： 15。
--------------------------------------------------
问题： 日本的首都是哪里？
回答： 日本的首都是东京。
--------------------------------------------------


### 复习：哪些结果得到了验证？

原有输出显示，重载后的三道题与保存前打印的回答一致。这支持“该 Adapter 可装回新的基础模型并完成这几道题的生成”，没有检验逐参数或逐 Logit 一致，也不是对泛化能力的检验。

把流程串起来：**模板和标签定义监督目标，Collator 定义一个 Batch 的形状，LoRA 定义哪些参数可训练，Trainer 定义何时更新、评估和保存。** 训练 6 条、验证 2 条的规模适合逐段理解这些机制；若要评价模型能力，还需要独立且更具代表性的测试集、基础模型对照与明确的评估指标。